In [1]:
import os
import librosa
import numpy as np
import random
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Layer, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences

from keras_efficient_kan import KANLinear

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
asthma_data = "./Datasets/Asthma"
copd_data = "./Datasets/COPD4"
healthy_data = "./Datasets/Healthy"

print("Asthma files:", len(os.listdir(asthma_data)))
print("COPD files:", len(os.listdir(copd_data)))
print("Healthy files:", len(os.listdir(healthy_data)))

Asthma files: 96
COPD files: 112
Healthy files: 112


In [3]:
def augment_audio(audio, sr):
    choice = random.randint(0, 2)
    if choice == 0: # Noise
        return audio + 0.005 * np.random.randn(len(audio))
    elif choice == 1: # Pitch
        return librosa.effects.pitch_shift(audio, sr=sr, n_steps=random.uniform(-2,2))
    else: # Speed
        return librosa.effects.time_stretch(audio, rate=random.uniform(0.9,1.1))

In [4]:
def extract_mfcc(audio, sr=22050):
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20, n_fft=2048, hop_length=512)
    return mfcc.T

In [5]:
def split_audio(audio, sr, win_sec=3, overlap_ratio=0.1):
    win_len = int(win_sec * sr)
    hop_len = int(win_len * (1-overlap_ratio))
    segments = []

    for start in range(0, len(audio)-win_len, hop_len):
        segments.append(audio[start:start+win_len])

    return segments


In [6]:
class_folders = {
    "Asthma": asthma_data,
    "COPD": copd_data,
    "Healthy": healthy_data
}

all_files = []
for label, folder in class_folders.items():
    for f in os.listdir(folder):
        if f.endswith('.wav'):
            all_files.append((os.path.join(folder, f), label))

train_files, test_files = train_test_split(
    all_files, test_size=0.2, stratify=[x[1] for x in all_files], random_state=42
)

In [7]:
def prepare_data(file_list, augment=False, aug_prob=0.3):
    X, y = [], []
    for path, label in file_list:
        try:
            audio, sr = librosa.load(path, sr=22050)
            #segments = split_audio(audio, sr)

            X.append(extract_mfcc(audio))
            y.append(label)

            if augment and random.random() < aug_prob:
                aug_seg = augment_audio(audio, sr)
                X.append(extract_mfcc(aug_seg))
                y.append(label)

        except Exception as e:
            print(f"Hata: {path} okunamadı. {e}")
    return X, y


In [8]:
print("Veriler işleniyor...")
X_train_raw, y_train_raw = prepare_data(train_files, augment=True, aug_prob=0.3)
X_test_raw, y_test_raw = prepare_data(test_files, augment=False)

Veriler işleniyor...


In [9]:
# Padding
X_train_pad = pad_sequences(X_train_raw, padding="post", dtype="float32")
X_test_pad = pad_sequences(X_test_raw, padding="post", dtype="float32", 
                            maxlen=X_train_pad.shape[1])

# Channel ekle
X_train_cnn = X_train_pad[..., np.newaxis]
X_test_cnn  = X_test_pad[..., np.newaxis]

# Label Encoding
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_raw)
y_test_enc  = le.transform(y_test_raw)


In [10]:
class NonlinearKAN(Layer):
    def __init__(self, units, hidden=16):
        super().__init__()
        self.units = units
        self.hidden = hidden

    def build(self, input_shape):
        self.mlps = []

        for _ in range(input_shape[-1]):
            self.mlps.append(
                tf.keras.Sequential([
                    tf.keras.layers.Dense(self.hidden, activation="swish"),
                    tf.keras.layers.Dense(self.units)
                ])
            )

    def call(self, x):
        outs = []
        for i in range(x.shape[-1]):
            xi = x[:, i:i+1]
            outs.append(self.mlps[i](xi))

        return tf.add_n(outs)


In [11]:
def build_nonlinear_kan(input_shape, layers):
    model = Sequential()
    model.add(Flatten(input_shape=input_shape))

    for u in layers:
        model.add(NonlinearKAN(u))
        model.add(tf.keras.layers.Activation("relu"))
        model.add(Dropout(0.3))

    model.add(Dense(3, activation="softmax"))

    model.compile(
        optimizer=Adam(1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


In [12]:


configs = {

    "1_layer": [[32],[64],[128],[256],[512]],

    "2_layer": [[32,64],[64,128],[128,256],[256,512]],

    "3_layer": [[32,64,128],[64,128,256],[128,256,512]]

}

In [13]:
results = []

for group, cfgs in configs.items():
    print(f"\n--- {group} Test Ediliyor ---")

    for filters in cfgs:
        model = build_nonlinear_kan(X_train_cnn.shape[1:], filters)

        model.fit(
            X_train_cnn, y_train_enc,
            validation_data=(X_test_cnn, y_test_enc),
            epochs=100,
            batch_size=32,
            verbose=0
        )

        loss, acc = model.evaluate(X_test_cnn, y_test_enc, verbose=0)
        print(f"Filters {filters} -> Acc: {acc:.4f}")

        results.append({"cfg": filters, "acc": acc})



--- 1_layer Test Ediliyor ---


c:\Users\MONSTER\Desktop\LungSoundAnlysis\LungSoundAnlysis\venv\lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


MemoryError: 

In [ ]:
best = max(results, key=lambda x: x["acc"])
print("\n🔥 EN İYİ MODEL 🔥")
print(best)



🔥 EN İYİ MODEL 🔥
{'cfg': [32, 64], 'acc': 0.657952070236206}
